<a href="https://colab.research.google.com/github/qmafaza/CI-Projek-BOA/blob/main/data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle

In [ ]:
!mv kaggle.json ~/.kaggle/kaggle.json

In [ ]:
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c new-york-city-taxi-fare-prediction

 99% 1.55G/1.56G [00:10<00:00, 37.7MB/s]
100% 1.56G/1.56G [00:10<00:00, 157MB/s] 


In [ ]:
!unzip /content/new-york-city-taxi-fare-prediction.zip -d /content/data

Archive:  /content/new-york-city-taxi-fare-prediction.zip
  inflating: /content/data/GCP-Coupons-Instructions.rtf  
  inflating: /content/data/sample_submission.csv  
  inflating: /content/data/test.csv  
  inflating: /content/data/train.csv  


Ambil 10% data random dari data train

In [ ]:
import random

input_file = "/content/data/train.csv"
output_file = "/content/data/train_0.1.csv"

sample_ratio = 0.1

with open(input_file, "r", encoding="utf-8") as src, open(output_file, "w", encoding="utf-8") as dst:
    header = src.readline()
    dst.write(header)

    for line in src:
        if random.random() < sample_ratio:
            dst.write(line)


In [ ]:
import pandas as pd

train_data = pd.read_csv('/content/data/train_0.1.csv')
test_data = pd.read_csv('/content/data/test.csv')

# Tujuan
Hitung distance (jarak) menggunakan rumus Haversine untuk setiap perjalanan di DataFrame 'train_data' dan 'test_data', membuat kolom baru 'distance' berdasarkan kolom 'pickup_latitude', 'pickup_longitude', 'dropoff_latitude', dan 'dropoff_longitude'

## Mendefinisikan rumus Haversine


In [ ]:
import numpy as np

def haversine_distance(pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude):
    # Earth's radius in kilometers
    R = 6371

    # Convert degrees to radians
    pickup_latitude_rad = np.radians(pickup_latitude)
    pickup_longitude_rad = np.radians(pickup_longitude)
    dropoff_latitude_rad = np.radians(dropoff_latitude)
    dropoff_longitude_rad = np.radians(dropoff_longitude)

    # Differences in coordinates
    dlat = dropoff_latitude_rad - pickup_latitude_rad
    dlon = dropoff_longitude_rad - pickup_longitude_rad

    # Haversine formula
    a = np.sin(dlat / 2)**2 + np.cos(pickup_latitude_rad) * np.cos(dropoff_latitude_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = R * c

    return distance

print("Haversine distance function defined.")

Haversine distance function defined.


## Menghitung jarak untuk Train Data

Menerapkan fungsi Haversine yang telah didefinisikan ke data train dan menyimpan hasilnya ke kolom 'distance'


In [ ]:
train_data['distance'] = haversine_distance(train_data['pickup_latitude'], train_data['pickup_longitude'], train_data['dropoff_latitude'], train_data['dropoff_longitude'])
print("Distance column added to train_data.")

Distance column added to train_data.


Lakukan hal yang sama pada data test


In [ ]:
test_data['distance'] = haversine_distance(test_data['pickup_latitude'], test_data['pickup_longitude'], test_data['dropoff_latitude'], test_data['dropoff_longitude'])
print("Distance column added to test_data.")

Distance column added to test_data.


## Menghilangkan data tidak valid

Meliputi:<br>
a. Nilai tarif negatif<br>
b. Nilai jarak lebih dari 8000 km.<br>
c. Lokasi pickup dan dropoff sama (jarak 0).<br>
d. Jarak > 50 km tetapi tarif < 10 (data outlier)

In [ ]:
print("Original shape of train_data:", train_data.shape)
print("Original shape of test_data:", test_data.shape)

# Filter negative fare amount (only for train_data)
train_data = train_data[train_data['fare_amount'] > 0]

# Filter distance > 8000 km
train_data = train_data[train_data['distance'] < 8000]
test_data = test_data[test_data['distance'] < 8000]

# Filter pickup and dropoff same location (distance == 0)
train_data = train_data[train_data['distance'] != 0]
test_data = test_data[test_data['distance'] != 0]

# Filter outliers: distance > 50 km but fare < 10 (only for train_data)
train_data = train_data[~((train_data['distance'] > 50) & (train_data['fare_amount'] < 10))]

print("Shape of train_data after filtering:", train_data.shape)
print("Shape of test_data after filtering:", test_data.shape)

print("Invalid and outlier data points filtered.")

Original shape of train_data: (5538470, 9)
Original shape of test_data: (9914, 8)
Shape of train_data after filtering: (5368620, 9)
Shape of test_data after filtering: (9829, 8)
Invalid and outlier data points filtered.


In [ ]:
print("First 5 rows of train_data with distance column:")
print(train_data.head())
print("\nFirst 5 rows of test_data with distance column:")
print(test_data.head())

First 5 rows of train_data with distance column:
                             key  fare_amount          pickup_datetime  \
0    2009-06-15 17:26:21.0000001          4.5  2009-06-15 17:26:21 UTC   
1    2012-04-08 07:30:50.0000002          5.3  2012-04-08 07:30:50 UTC   
2    2011-04-05 17:11:05.0000001          7.7  2011-04-05 17:11:05 UTC   
3  2009-07-22 16:08:00.000000163          5.3  2009-07-22 16:08:00 UTC   
4  2011-06-28 19:47:00.000000168          4.5  2011-06-28 19:47:00 UTC   

   pickup_longitude  pickup_latitude  dropoff_longitude  dropoff_latitude  \
0        -73.844311        40.721319         -73.841610         40.712278   
1        -73.996335        40.737142         -73.980721         40.733559   
2        -74.001821        40.737547         -73.998060         40.722788   
3        -73.981060        40.737690         -73.994177         40.728412   
4        -73.988893        40.760160         -73.986445         40.757857   

   passenger_count  distance  
0           

## Menyimpan data train yang sudah dipreprocessing ke file CSV

In [ ]:
train_data.to_csv('/content/data/train_preprocessed.csv', index=False)
print("Preprocessed train_data saved to /content/data/train_preprocessed.csv")

Preprocessed train_data saved to /content/data/train_preprocessed.csv


## Menyimpan data test yang sudah dipreprocessing ke file CSV

In [ ]:
test_data.to_csv('/content/data/test_preprocessed.csv', index=False)
print("Preprocessed test_data saved to /content/data/test_preprocessed.csv")

Preprocessed test_data saved to /content/data/test_preprocessed.csv
